# End of week 1 exercise

A tool that takes a technical question and responds with an explanation,
using both the OpenAI API (gpt-5-mini, streamed) and a local Ollama model (llama3.2).

In [1]:
# imports

import os
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import Markdown, display, update_display


In [2]:
# constants

MODEL_GPT = 'gpt-5-mini'
MODEL_LLAMA = 'llama3.2'
OLLAMA_BASE_URL = 'http://localhost:11434/v1'

In [3]:
# set up environment

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if not api_key:
    print("No OPENAI_API_KEY found - check your .env file")
elif not api_key.startswith('sk-'):
    print("An API key was found, but it doesn't start with sk- ; check it")
else:
    print("OpenAI API key found and looks good")

# Client for OpenAI's cloud API
openai = OpenAI()

# Client for local Ollama - it exposes an OpenAI-compatible endpoint,
# so we reuse the same library and just point it at localhost.
ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')

OpenAI API key found and looks good


In [4]:
# prompts

system_prompt = (
    "You are a helpful technical tutor who answers questions about Python code, "
    "software engineering, data science and LLMs. Explain clearly and step by step, "
    "show small examples where useful, and respond in Markdown."
)

def user_prompt_for(question):
    return "Please give a detailed explanation to the following question:\n\n" + question

def messages_for(question):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_for(question)},
    ]

In [5]:
# here is the question; type over this to ask something new

question = """
Please explain what this code does and why:
yield from {book.get("author") for book in books if book.get("author")}
"""

In [6]:
# helper: stream a response from any OpenAI-compatible client into the notebook

def stream_answer(client, model, question):
    stream = client.chat.completions.create(
        model=model,
        messages=messages_for(question),
        stream=True,
    )
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)
    return response

In [ ]:
# Get gpt-5-mini to answer, with streaming

gpt_answer = stream_answer(openai, MODEL_GPT, question)

Short answer
- That line, inside a generator function, yields each unique, truthy author found in the iterable books. It builds a set of authors and then yields each element of that set (order not guaranteed).

Detailed step-by-step explanation
1. The expression inside the braces is a set comprehension:
   {book.get("author") for book in books if book.get("author")}
   - For each book in books it calls book.get("author").
   - If that call returns a truthy value (not None, not empty string, not 0, etc.) the value is included in the set.
   - Because it is a set comprehension, duplicates are removed (only one copy of each author remains).

2. yield from <iterable>
   - The yield from statement iterates over the iterable on its right-hand side and yields each element to the caller of the generator.
   - Here the iterable is the set created by the comprehension. So the generator yields each unique author from that set.

3. Combined effect
   - The line yields each unique, non-falsy author from books, in an arbitrary order (sets do not preserve insertion order).
   - Example behavior: if books contains multiple dictionaries with the same author, that author is yielded only once.

Example
Given:
books = [
    {"title": "A", "author": "Alice"},
    {"title": "B", "author": "Bob"},
    {"title": "C", "author": "Alice"},
    {"title": "D"}  # no author
]

If used inside a generator:
def gen():
    yield from {book.get("author") for book in books if book.get("author")}

list(gen()) might produce ['Bob', 'Alice'] (order can vary).

Important notes and caveats
- Duplicate removal: Using a set removes duplicates. If you want to keep duplicates (i.e., yield every author occurrence), use a generator expression instead:
  yield from (book.get("author") for book in books if book.get("author"))

- Order: Sets are unordered (iteration order is arbitrary). If you want unique authors but preserve the first-seen order, use a dict (Python 3.7+ keeps insertion order) or use OrderedDict:
  yield from dict.fromkeys(
      (book.get("author") for book in books if book.get("author"))
  )
  This yields unique authors in first-seen order.

- Double call to book.get: The comprehension as written calls book.get("author") twice for each book (once in the expression, once in the if). That can be avoided with the walrus operator (Python 3.8+):
  yield from {author for book in books if (author := book.get("author"))}

  Or with an inner generator:
  yield from {a for a in (book.get("author") for book in books) if a}

- Eager vs lazy: The set comprehension builds the entire set eagerly (all authors are collected before any are yielded). If the books iterable is large and you prefer streaming/lazy behavior, use an appropriate generator expression (but note that generators will not deduplicate automatically).

Summary / recommended patterns
- If you want unique authors and order doesn't matter: the original line is fine (but consider using walrus to avoid double get).
- If you want unique authors and preserve first-seen order: use dict.fromkeys(...) on a generator.
- If you want to yield every author (including duplicates) lazily: use yield from (book.get('author') for book in books if book.get('author')).

If you want, I can show the alternative versions with full generator functions and timings to illustrate the differences.

In [8]:
# Get Llama 3.2 to answer (runs locally via Ollama - make sure `ollama serve` is running)

llama_answer = stream_answer(ollama, MODEL_LLAMA, question)

**Understanding the Code**
==========================

The given code snippet is written in Python and utilizes a feature called generator expressions, `yield from`, and dictionary iteration. We'll break it down step by step to understand what it does and why.

```python
from typing import Generator, Dict, Any

# Assuming these are defined elsewhere in the code
books: list[Dict[str, Any]] = [...]  # List of dictionaries
def get_authors(books: list[Dict[str, Any]]) -> Generator[str, None, None]:
    for book in books:
        yield book.get("author")
```

**Explanation**
--------------

### 1. Dictionary Iteration

```python
for book in books:
```

This is a standard for loop to iterate through the `books` list.

### 2. Generator Expression

```python
(book.get("author") for book in books)
```

This is a generator expression, which is a concise way to create a generator. In this case, it's iterating through the `books` list and yielding the value of the `"author"` key in each dictionary. If the key does not exist, `get()` returns `None`.

### 3. `yield from`

```python
yield from {book.get("author") for book in books if book.get("author")}
```

Here's where things get interesting. `yield from` is a special syntax used to delegate execution to a subgenerator. A subgenerator is another generator that can be used to yield values.

In this case, we have a dictionary comprehension that yields the value of the `"author"` key. However, instead of using another generator expression, we use `yield from`. This allows us to delegate the execution to the subgenerator.

The `yield from` syntax is equivalent to:

```python
for author in {book.get("author") for book in books if book.get("author")}: 
    yield author
```

This is more efficient because it avoids creating an intermediate list and can handle very large data sets.

### 4. Generator Yield

We've created a generator that yields values, just like a regular function would using `yield`.

### Why `yield from`?

Using `yield from` is more efficient than creating a new generator each time because it avoids the overhead of creating and destroying a new generator object.

Additionally, `yield from` allows us to simplify our code by delegating the execution to a subgenerator. This makes it easier to read and understand the code.

**Conclusion**
----------

In summary, this code creates a generator that yields the values of the `"author"` key from each dictionary in the `books` list. It does this using a generator expression and the `yield from` syntax to delegate execution to a subgenerator, which is more efficient than creating a new generator object.